# Fix Recommendation Agent - Logic Explained

## 🎯 **Overview**

The Fix Recommendation Agent is an intelligent system that analyzes data quality issues and generates actionable fix suggestions. It uses multiple strategies based on the type of issue detected.

## 🏗️ **Architecture**

```
Fix Recommendation Agent
    ↓
Receives: Issues + Anomalies
    ↓
Categorizes by type:
    ├─→ Completeness Issues → Imputation strategies
    ├─→ Conformity Issues → Format corrections
    ├─→ Uniqueness Issues → Duplicate removal
    └─→ Anomalies → Root cause investigation
    ↓
Generates Recommendations:
    - Fix strategy
    - Confidence score (0-100%)
    - Actionability (auto-fix vs review)
    - Risk assessment
    - Prerequisites
```

---

## 📋 **Step-by-Step Logic Flow**

### **1. Input Processing** (`execute` method)

```python
# Agent receives either:
- profile_results (from Profiler Agent)
- anomaly_results (from Anomaly Detection Agent)

# OR directly:
- issues (list of Issue objects)
- anomalies (list of Anomaly objects)
```

**Key Logic:**
- Validates inputs (must have either issues or anomalies)
- Extracts dataset profile metadata
- Calls appropriate recommendation methods

---

## 🔍 **2. Issue Type Routing**

The agent routes each issue to a specific handler based on its type:

### **Completeness Issues**
→ `_recommend_completeness_fix()`
- Missing values in columns
- Strategy: Imputation (fill missing values)

### **Conformity Issues**
→ `_recommend_conformity_fix()`
- Values don't match expected format
- Strategy: Format correction

### **Uniqueness Issues**
→ `_recommend_uniqueness_fix()`
- Duplicate values detected
- Strategy: Duplicate removal

### **Anomalies**
→ `_recommend_for_anomaly()`
- Statistical outliers from baseline
- Strategy: Root cause investigation

---

## 🧩 **Detailed Logic: Completeness Issues**

### **Strategy Selection Flow**

```
Missing Values Detected
    ↓
Has most_common_values? 
    ├─ Yes → Calculate mode coverage
    │   ├─ Coverage > 20%? 
    │   │   ├─ Yes → MODE IMPUTATION (confidence = coverage%)
    │   │   └─ No → Try defaults
    │   └─ Coverage > 70%? 
    │       └─ Yes → Higher confidence (up to 70%)
    │
    └─ No → Try column defaults
        ├─ Default exists? 
        │   ├─ Yes → DEFAULT IMPUTATION (confidence = 60%)
        │   └─ No → FLAG FOR REVIEW (confidence = 50%)
```

### **Example Logic:**

```python
# Example 1: Entity Type Code has 6 missing values
# Most common value: "1.0" (appears in 72% of records)
→ Strategy: mode_imputation
→ Confidence: 70.0% (min(0.7, 0.72))
→ Suggested Value: "1.0"
→ Actionable: False (70% < 85% threshold)

# Example 2: Employer Identification Number has 78 missing values
# Most common value: "<UNAVAIL>" (appears in 22% of records)
→ Strategy: mode_imputation
→ Confidence: 22.0% (min(0.7, 0.22))
→ Suggested Value: "<UNAVAIL>"
→ Actionable: False (22% < 85% threshold)

# Example 3: Last Update Date has 6 missing values
# No clear mode, but has default
→ Strategy: default_imputation
→ Confidence: 60%
→ Suggested Value: "10/31/2025" (current date)
→ Actionable: False (60% < 85% threshold)

# Example 4: Provider City has 6 missing values
# No clear mode, no default
→ Strategy: flag_for_review
→ Confidence: 50%
→ Suggested Value: None
→ Actionable: False
→ Rationale: "Manual review recommended"
```

### **Actionability Threshold**

```python
SAFE_IMPUTATION_CONFIDENCE = 0.85  # 85%

if confidence >= 0.85:
    actionable = True  # Safe to auto-fix
    impact = "low" if percentage < 5 else "medium"
else:
    actionable = False  # Requires human review
    impact = "high"
```

---

## 🔧 **Detailed Logic: Conformity Issues**

### **Strategy Selection Flow**

```
Format Violation Detected
    ↓
Get validation rule for column
    ↓
Analyze violation pattern by rule type:
    ├─ regex → Pattern mismatch
    │   └─ Suggest: "Check pattern", "Verify source"
    │
    ├─ length → Value too short/long
    │   ├─ Too short → "Truncate short values"
    │   └─ Too long → "Truncate long values"
    │
    └─ enum → Invalid value
        └─ Suggest: "Use one of: [allowed values]"
    ↓
Generate rationale from patterns
```

### **Example Logic:**

```python
# Example: Postal Code has 7 format violations
# Rule type: regex
→ Strategy: format_correction
→ Confidence: 80% (if specific fixes found)
→ Rationale: "7 format violations detected. Common fixes: Check regex pattern, Verify data source"
→ Actionable: False (80% < 90% threshold for format correction)

# Example: Primary Taxonomy Switch has 2 violations
# Rule type: enum (values must be "Y" or "N")
→ Strategy: format_correction
→ Confidence: 80%
→ Rationale: "Ensure value is one of: Y, N"
→ Actionable: False (80% < 90% threshold)
```

### **Actionability Threshold**

```python
SAFE_FORMAT_CORRECTION_CONFIDENCE = 0.90  # 90%

if confidence >= 0.90 AND specific_fixes_identified:
    actionable = True
else:
    actionable = False  # Format correction is risky
```

---

## 🔄 **Detailed Logic: Uniqueness Issues**

### **Strategy Selection Flow**

```
Duplicate Values Detected
    ↓
Is it NPI column?
    ├─ Yes → CRITICAL ISSUE
    │   └─ Strategy: remove_duplicates
    │       Confidence: 95%
    │       Impact: HIGH
    │       Actionable: False (always requires review)
    │
    └─ No → Standard duplicates
        └─ Strategy: remove_duplicates
            Confidence: 80%
            Impact: high
            Actionable: False
```

### **Example Logic:**

```python
# Example: NPI has duplicates
→ Strategy: remove_duplicates
→ Confidence: 95%
→ Impact: "high"
→ Actionable: False (NPI duplicates are critical)
→ Rationale: "Critical: NPI must be unique. X duplicates detected"
→ Risk: "HIGH RISK: Duplicate NPIs may indicate data integrity issues"
```

### **NPI Uniqueness Special Handling**

```python
# NPI duplicates are ALWAYS flagged for review
# Because they are:
- Critical identifiers
- May indicate data quality issues
- Require careful investigation
```

---

## 📊 **Detailed Logic: Anomaly Remediation**

### **Strategy Selection Flow**

```
Anomaly Detected (statistical outlier)
    ↓
Analyze direction:
    ├─ delta < 0 → Quality degraded
    │   └─ Description: "Metric dropped X% from baseline"
    │
    └─ delta > 0 → Quality improved unexpectedly
        └─ Description: "Metric increased X% from baseline"
    ↓
Assess severity:
    ├─ severity == "high" AND z_score > 3
    │   └─ Confidence: 90%
    │
    ├─ severity == "high"
    │   └─ Confidence: 70%
    │
    └─ Otherwise
        └─ Confidence: 50%
    ↓
Strategy: investigate_root_cause
Actionable: False (always requires investigation)
```

### **Example Logic:**

```python
# Example: Completeness dropped 15% unexpectedly
→ Strategy: investigate_root_cause
→ Description: "Completeness dropped 15.0% from baseline"
→ Rationale: "Degradation detected: completeness fell from 85.0% to 70.0%"
→ Confidence: 70% (if severity is high)
→ Actionable: False (always requires investigation)
→ Prerequisites: ["backup_dataset", "investigate_upstream_source", 
                  "check_data_ingestion_process"]
```

---

## 🎚️ **Confidence Scoring System**

### **Scale: 0.0 to 1.0 (0% to 100%)**

| Confidence | Meaning | Auto-Fix Possible? |
|------------|---------|--------------------|
| 0.95-1.0 | Very High | Yes (for duplicates) |
| 0.85-0.95 | High | Yes (for imputation) |
| 0.70-0.85 | Medium-High | No (review recommended) |
| 0.50-0.70 | Medium | No (review required) |
| < 0.50 | Low | No (manual review required) |

### **Confidence Calculation Examples**

**Mode Imputation:**
```python
coverage = (most_common_count / total_count) * 100
confidence = min(0.7, coverage / 100)
# Example: 72% coverage → 70% confidence
```

**Default Imputation:**
```python
confidence = 0.6  # Fixed 60% for defaults
```

**Format Correction:**
```python
confidence = 0.7  # Base 70%
if specific_fixes_found:
    confidence = 0.8  # Raise to 80%
```

**Anomaly Remediation:**
```python
if severity == "high" and z_score > 3:
    confidence = 0.9
elif severity == "high":
    confidence = 0.7
else:
    confidence = 0.5
```

---

## 🚨 **Actionability Rules**

### **Safe Auto-Fix Thresholds**

```python
SAFE_IMPUTATION_CONFIDENCE = 0.85          # 85% confidence
SAFE_FORMAT_CORRECTION_CONFIDENCE = 0.90   # 90% confidence
SAFE_DUPLICATE_REMOVAL_CONFIDENCE = 0.95   # 95% confidence
```

### **When Can a Fix Be Auto-Applied?**

**Completeness (Imputation):**
- ✅ Coverage >= 85% OR default exists with 85%+ confidence
- ✅ Impact is low (< 5% of records affected)
- ✅ Suggested value is validated

**Conformity (Format Correction):**
- ✅ Confidence >= 90%
- ✅ Specific fix pattern identified
- ✅ Low risk of data corruption

**Uniqueness (Duplicate Removal):**
- ✅ Never auto-applied (always requires review)
- ⚠️ Critical for NPI columns
- ⚠️ May indicate systematic issues

**Anomalies (Root Cause Investigation):**
- ✅ Never auto-applied (always requires investigation)
- ⚠️ Indicates systematic issues
- ⚠️ Requires upstream investigation

---

## 🎯 **Risk Assessment Logic**

### **Risk Levels**

| Level | Meaning | Recommendation |
|-------|---------|----------------|
| **Low** | Safe to apply | Auto-fix when confidence >= threshold |
| **Medium** | Needs verification | Review before applying |
| **High** | Requires expert input | Manual review required |

### **Risk Calculation**

**Completeness:**
```python
if actionable and percentage < 5:
    risk = "Low risk: value imputation based on data patterns"
elif actionable:
    risk = "Medium risk: value imputation based on data patterns"
else:
    risk = "Medium risk: requires domain expert review"
```

**Conformity:**
```python
risk = "Data transformation required"  # Always medium
```

**Uniqueness:**
```python
if column_name == "NPI":
    risk = "HIGH RISK: Duplicate NPIs may indicate data integrity issues"
else:
    risk = "Manual investigation required"
```

**Anomalies:**
```python
risk = f"Systematic issue detected (z-score: {z_score:.2f}, severity: {severity})"
```

---

## 📝 **Prerequisites System**

Every recommendation includes prerequisites that must be completed before applying the fix:

### **Common Prerequisites**

```python
# Always included
prerequisites = ["backup_dataset"]

# Type-specific
completeness: ["validate_suggested_value"]
conformity: ["validate_format_corrections"]
uniqueness: ["investigate_duplicate_source"]
anomalies: ["investigate_upstream_source", "check_data_ingestion_process"]
```

### **Default Values Configuration**

The agent maintains a dictionary of default values for specific columns:

```python
defaults = {
    "Entity Type Code": "1",
    "Last Update Date": datetime.now().strftime("%m/%d/%Y"),
}
```

---

## 🔄 **Complete Example Flow**

### **Input: Issue from Profiler**

```python
Issue(
    issue_id="abc-123",
    dataset_name="npidata_sample_100",
    column_name="Provider City Name",
    issue_type="completeness",
    count=6,
    percentage=6.0,
    description="Required field has 6 null values"
)
```

### **Processing:**

1. **Route to completeness handler**
2. **Check for mode value**
   - No clear mode found
3. **Check for defaults**
   - No default for "Provider City Name"
4. **Fallback to flag_for_review**
   - Confidence: 50%
   - Strategy: flag_for_review

### **Output: Recommendation**

```python
FixRecommendation(
    recommendation_id="rec-xyz",
    issue_id="abc-123",
    dataset_name="npidata_sample_100",
    column_name="Provider City Name",
    issue_type="completeness",
    fix_strategy="flag_for_review",
    fix_description="Fill 6 missing values in Provider City Name",
    confidence_score=0.5,  # 50%
    estimated_impact="high",
    actionable=False,  # Cannot auto-fix
    suggested_value=None,  # No safe suggestion
    rationale="Missing values detected in Provider City Name - manual review recommended",
    prerequisites=["backup_dataset", "validate_suggested_value"],
    risk_assessment="Medium risk: requires domain expert review"
)
```

---

## 🎛️ **Configuration**

From `config/agents.yaml`:

```yaml
fix_recommendation:
  confidence_threshold: 0.75  # General threshold (not used for auto-fix)
  enable_auto_suggestions: true
  enable_llm_fixes: false  # Future: LLM-powered fixes
```

**Note:** The agent uses hardcoded thresholds (`SAFE_IMPUTATION_CONFIDENCE`, etc.) for actual auto-fix decisions. The `confidence_threshold` in config is for general categorization but doesn't override the safety thresholds.

---

## 💡 **Key Design Principles**

1. **Safety First:** Auto-fix only when confidence is very high
2. **Explainability:** Every recommendation includes rationale
3. **Prerequisites:** Always require backups before changes
4. **Risk Assessment:** Quantify the impact of each fix
5. **Context-Aware:** Different strategies for different column types
6. **Human-in-the-Loop:** Most fixes require review
7. **Pattern-Based:** Use data patterns (mode, common values) when available
8. **Defaults Fallback:** Use sensible defaults when applicable

---

## 🎯 **Summary**

The Fix Recommendation Agent is a **smart, safety-first system** that:

- ✅ Analyzes issues by type (completeness, conformity, uniqueness, anomalies)
- ✅ Uses multiple strategies (imputation, format correction, duplicate removal, root cause investigation)
- ✅ Scores confidence based on data patterns
- ✅ Flags auto-fixable vs review-required recommendations
- ✅ Provides detailed rationale and risk assessment
- ✅ Requires prerequisites before applying fixes
- ✅ Emphasizes human oversight

**The agent's philosophy:** "Better to recommend review than to auto-fix incorrectly."



In [ ]:
import pandas as 